In [1]:
%reload_ext autoreload
%autoreload 2

import json
from IPython.display import display

from src.config import CONFIG_DIR
from src.data_interface import loader
from src.data_interface.validation import _field_key, load_schema, validate_table
from src.pipeline.context import RunContext
from src.utilities.visualize import kepler_visualize

context = RunContext("TestScenario")
# Read status at display time so a preparation run is visible without restarting.
def print_status(table):
    statuses = json.loads((CONFIG_DIR / "_status.json").read_text(encoding="utf-8"))
    status = statuses.get(table)
    if status is None:
        print(f"{table}: No status recorded")
        return
    definition, = [tables[table] for tables in load_schema().values() if table in tables]
    allowed = {_field_key(field["Name"]) for field in definition["Fields"]}
    obsolete = {name for fields in status.values() for name in fields if _field_key(name) not in allowed}
    assert not obsolete, (
        f"{table} status contains obsolete fields: {sorted(obsolete)}. "
        "Rerun create_temp_schema.ipynb to refresh staged tables and status."
    )
    print(f"{table}: {len(status['completed_fields'])} completed fields; "
          f"{len(status['fields_with_missing_values'])} fields with missing values; "
          f"{len(status['problematic_fields'])} fields needing review")
    # Keep completed field names in JSON; show percentages and derivation notes here.
    display({key: value for key, value in status.items() if key != "completed_fields"})


c:\Users\lij11\AppData\Local\anaconda3\envs\geospatial\Lib\site-packages\keplergl\keplergl.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_string


In [2]:
# TAZs
print_status("TAZs")
tazs = loader.load(context, "TAZs", category="PreprocessedConstants")
tazs.head()


TAZs: 3 completed fields; 3 fields with missing values; 0 fields needing review


{'fields_with_missing_values': {'base_year_population': 92.6136,
  'base_year_workers': 92.6136,
  'base_year_jobs': 92.6136},
 'problematic_fields': {}}

,subregion_ids,corridor_ids,geometry,base_year_population,base_year_workers,base_year_jobs
taz_id,,,,,,
1,"[CITY_4835000, COUNTY_48201, PCT_48201_1]",[],"POLYGON ((-95.37331 29.75987, -95.37296 29.759...",NaN,NaN,NaN
2,"[CITY_4835000, COUNTY_48201, PCT_48201_1]",[],"POLYGON ((-95.37127 29.75859, -95.37121 29.758...",NaN,NaN,NaN
3,"[CITY_4835000, COUNTY_48201, PCT_48201_1]",[],"POLYGON ((-95.37336 29.75432, -95.37337 29.754...",NaN,NaN,NaN
4,"[CITY_4835000, COUNTY_48201, PCT_48201_1]",[CORR_IH0045-KG],"POLYGON ((-95.37277 29.75773, -95.37272 29.757...",NaN,NaN,NaN
5,"[CITY_4835000, COUNTY_48201, PCT_48201_1]",[],"POLYGON ((-95.37138 29.75524, -95.37213 29.755...",NaN,NaN,NaN


In [3]:
# Corridors
print_status("Corridors")
corridors = loader.load(context, "Corridors", category="PreprocessedConstants")
corridors.head()


Corridors: 5 completed fields; 0 fields with missing values; 0 fields needing review


{'fields_with_missing_values': {}, 'problematic_fields': {}}

,corridor_name,road_category,txdot_rte_name,length_miles,geometry
corridor_id,,,,,
CORR_FM1774-KG,FM 1774,Uncategorized,FM1774-KG,9.909,"MULTILINESTRING ((-95.75423 30.21124, -95.7491..."
CORR_FM3005-KG,FM 3005,Uncategorized,FM3005-KG,9.252,"LINESTRING (-94.82628 29.26653, -94.83033 29.2..."
CORR_TL0011-KG,Fort Bend Parkway,Toll,TL0011-KG,8.194,"LINESTRING (-95.48114 29.6408, -95.48151 29.64..."
CORR_BS0035C-KG,Grand Parkway,Toll (Future),BS0035C-KG,3.962,"LINESTRING (-95.24534 29.45594, -95.2453 29.45..."
CORR_FM0646-KG,Grand Parkway,Toll (Future),FM0646-KG,13.132,"LINESTRING (-94.99512 29.49878, -95.00038 29.4..."


In [4]:
# SubRegions
print_status("SubRegions")
subregions = validate_table(context, "PreprocessedConstants", "SubRegions")
subregions.head()


SubRegions: 3 completed fields; 1 fields with missing values; 0 fields needing review


{'fields_with_missing_values': {'fips': 24.5283}, 'problematic_fields': {}}

,subregion_type,subregion_name,geometry,fips
subregion_id,,,,
CITY_4801696,city,Aldine,"POLYGON ((-95.40871 29.9137, -95.4087 29.91312...",4801696
CITY_4802272,city,Alvin,"MULTIPOLYGON (((-95.22638 29.50177, -95.22652 ...",4802272
CITY_4803072,city,Ames,"POLYGON ((-94.7535 30.05349, -94.75354 30.0494...",4803072
CITY_4803144,city,Anahuac,"POLYGON ((-94.69031 29.74681, -94.69044 29.746...",4803144
CITY_4803264,city,Angleton,"MULTIPOLYGON (((-95.45983 29.16487, -95.45921 ...",4803264


In [5]:
# Cells
print_status("Cells")
cells = loader.load(context, "Cells", category="PreprocessedConstants")
cells.head()


Cells: 9 completed fields; 5 fields with missing values; 0 fields needing review


{'fields_with_missing_values': {'base_year_population': 92.2724,
  'base_year_households': 92.2724,
  'base_year_workers': 92.2724,
  'base_year_jobs': 92.2724,
  'base_year_imperviousness': 92.2763},
 'problematic_fields': {}}

,area_sqft,base_year_contains_retail,base_year_contains_recreation,base_year_contains_healthcare,flood_exposure,taz_ids,subregion_ids,corridor_ids,base_year_population,base_year_households,base_year_workers,base_year_jobs,base_year_imperviousness
cell_id,,,,,,,,,,,,,
89446124823ffff,1260883.555955,False,False,False,False,[4911],"[COUNTY_48167, PCT_48167_2, UNINC_48167]",[],NaN,NaN,NaN,NaN,NaN
89446124827ffff,1260910.703652,False,False,False,False,[4911],"[COUNTY_48167, PCT_48167_2, UNINC_48167]",[],NaN,NaN,NaN,NaN,NaN
8944612482bffff,1260833.482249,False,False,False,False,[4911],"[COUNTY_48167, PCT_48167_2, UNINC_48167]",[],NaN,NaN,NaN,NaN,NaN
8944612482fffff,1260860.636557,False,False,False,False,[4911],"[COUNTY_48167, PCT_48167_2, UNINC_48167]",[],NaN,NaN,NaN,NaN,NaN
89446124833ffff,1260906.47145,False,False,False,False,[4911],"[COUNTY_48167, PCT_48167_2, UNINC_48167]",[],NaN,NaN,NaN,NaN,NaN


In [6]:
# BaseHighwayLinks
print_status("BaseHighwayLinks")
base_highway_links = validate_table(context, "PreprocessedConstants", "BaseHighwayLinks")
base_highway_links.head()


BaseHighwayLinks: 5 completed fields; 20 fields with missing values; 4 fields needing review


{'fields_with_missing_values': {'base_year_lanes_count': 1.5498,
  'base_year_capacity': 1.5498,
  'base_year_free_flow_speed_mph': 1.5498,
  'base_year_cross_section': 1.5498,
  'base_year_facility_type': 1.5498,
  'base_year_area_access_type': 2.7906,
  'base_year_hin_percent': 1.5498,
  'is_bridge': 1.5498,
  'base_year_pavement_score': 69.1421,
  'base_year_bridge_score': 91.7123,
  'base_year_vulnerability_score': 32.21,
  'base_year_criticality_score': 32.2009,
  'base_year_srts_status': 1.5498,
  'is_in_1_mile_school_buffer': 1.5498,
  'is_in_100_year_flood_plain': 1.5498,
  'is_designated_freight_network': 1.5498,
  'cell_id': 1.5544,
  'corridor_id': 85.263,
  'taz_id': 1.5589,
  'base_year_roadway_class': 100.0},
 'problematic_fields': {'base_year_facility_type': 'tentative facility crosswalk; confirm NHS membership',
  'base_year_hin_percent': 'Segments are ranked by EPDO density \\(EPDO points per mile per year\\) in descending order\\. The HIN is defined as the set of top\

,node_1_id,node_2_id,base_year_distance_miles,base_year_lanes_count,base_year_capacity,base_year_free_flow_speed_mph,base_year_cross_section,base_year_facility_type,base_year_area_access_type,base_year_hin_percent,...,base_year_criticality_score,is_in_1_mile_school_buffer,base_year_srts_status,is_designated_freight_network,taz_id,cell_id,corridor_id,subregion_ids,geometry,base_year_roadway_class
link_id,,,,,,,,,,,,,,,,,,,,,
5809-26997,5809,26997,1.24212,4,15300.0,60.0,Undivided,Non Interstate NHS,Rural Unrestricted,0.0,...,0.179543,False,False,True,4031,89446d1103bffff,CORR_SH0006-KG,"[COUNTY_48473, PCT_48473_1, UNINC_48473]","LINESTRING (-96.05483 30.23059, -96.05801 30.2...",<NA>
5810-36183,5810,36183,1.12541,2,3700.0,45.0,Undivided,Other,Rural Unrestricted,0.0,...,0.0,False,False,False,3999,89446d1a24bffff,<NA>,"[COUNTY_48473, PCT_48473_2, UNINC_48473]","LINESTRING (-95.95032 30.23693, -95.94159 30.2...",<NA>
5811-27007,5811,27007,0.35634,2,4400.0,53.0,Undivided,Other,Rural Unrestricted,0.0,...,0.0,False,False,True,4001,89446dcca2fffff,CORR_FM1774-KG,"[COUNTY_48473, PCT_48473_2, UNINC_48473]","LINESTRING (-95.82235 30.24426, -95.8184 30.24...",<NA>
5812-26990,5812,26990,3.32934,2,34500.0,64.0,Divided,Interstate,Rural Restricted,0.0,...,0.029193,False,False,True,4043,89446d124d7ffff,CORR_US0290-KG,"[COUNTY_48473, PCT_48473_1, UNINC_48473]","LINESTRING (-96.18691 30.12929, -96.13169 30.1...",<NA>
5813-24377,5813,24377,0.3301,2,4400.0,53.0,Undivided,Other,Rural Unrestricted,0.0,...,0.000238,False,False,False,4040,89446d89517ffff,<NA>,"[COUNTY_48015, COUNTY_48473, PCT_48015_1, PCT_...","LINESTRING (-96.11122 30.04338, -96.10727 30.0...",<NA>


In [7]:
# BaseYearTDM
print_status("BaseYearTDM")
base_year_tdm = validate_table(context, "PreprocessedConstants", "BaseYearTDM")
base_year_tdm.head()

BaseYearTDM: 12 completed fields; 8 fields with missing values; 8 fields needing review


{'fields_with_missing_values': {'base_year_medium_truck_ratio_am': 100.0,
  'base_year_medium_truck_ratio_pm': 100.0,
  'base_year_medium_truck_ratio_md': 100.0,
  'base_year_medium_truck_ratio_ov': 100.0,
  'base_year_heavy_truck_ratio_am': 100.0,
  'base_year_heavy_truck_ratio_pm': 100.0,
  'base_year_heavy_truck_ratio_md': 100.0,
  'base_year_heavy_truck_ratio_ov': 100.0},
 'problematic_fields': {'base_year_medium_truck_ratio_am': '100% missing values; nullable placeholder',
  'base_year_medium_truck_ratio_pm': '100% missing values; nullable placeholder',
  'base_year_medium_truck_ratio_md': '100% missing values; nullable placeholder',
  'base_year_medium_truck_ratio_ov': '100% missing values; nullable placeholder',
  'base_year_heavy_truck_ratio_am': '100% missing values; nullable placeholder',
  'base_year_heavy_truck_ratio_pm': '100% missing values; nullable placeholder',
  'base_year_heavy_truck_ratio_md': '100% missing values; nullable placeholder',
  'base_year_heavy_truck_rat

,base_year_total_daily_vmt,base_year_total_daily_truck_vmt,base_year_total_daily_volume,base_year_travel_time_minutes_am,base_year_travel_time_minutes_md,base_year_travel_time_minutes_pm,base_year_travel_time_minutes_ov,base_year_volume_am,base_year_volume_md,base_year_volume_pm,base_year_volume_ov,base_year_medium_truck_ratio_am,base_year_medium_truck_ratio_pm,base_year_medium_truck_ratio_md,base_year_medium_truck_ratio_ov,base_year_heavy_truck_ratio_am,base_year_heavy_truck_ratio_pm,base_year_heavy_truck_ratio_md,base_year_heavy_truck_ratio_ov
link_id,,,,,,,,,,,,,,,,,,,
5809-26997,14902.34668,636.520081,11997.508789,1.268953,1.312021,1.386258,1.243405,1638.198608,4307.278,3531.1018,2520.9306,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
5810-36183,749.373352,10.918503,665.867004,1.500592,1.50063,1.501118,1.500554,87.6022,207.8579,207.772247,162.634644,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
5811-27007,872.205811,17.424135,2447.678711,0.405246,0.404587,0.407628,0.403456,456.143738,803.845154,666.2462,521.443542,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
5812-26990,51812.921875,1966.111084,15562.520508,3.125692,3.134768,3.142623,3.12156,2063.141846,5672.4419,4310.654297,3516.2826,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
5813-24377,964.066589,7.880312,2920.529053,0.37886,0.375578,0.381051,0.37376,625.8097,937.8156,797.804932,559.0987,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [8]:
# BikePedAccessibilityMatrix
print_status("BikePedAccessibilityMatrix")
bike_ped_accessibility = loader.load(context, "BikePedAccessibilityMatrix", category="PreprocessedConstants")
bike_ped_accessibility.head()

BikePedAccessibilityMatrix: 6 completed fields; 0 fields with missing values; 0 fields needing review


{'fields_with_missing_values': {}, 'problematic_fields': {}}

,bike_30_min_cell_ids,buffer_1_mile_cell_ids,buffer_quarter_mile_cell_ids,walk_15_min_cell_ids,buffer_half_mile_cell_ids
cell_id,,,,,
89446124823ffff,"[89446124823ffff, 89446c51003ffff, 89446c51007...","[89446124823ffff, 89446124827ffff, 8944612482b...","[89446124823ffff, 89446124827ffff, 8944612482b...",[89446124823ffff],"[89446124823ffff, 89446124827ffff, 8944612482b..."
89446124827ffff,"[89446124827ffff, 89446c51003ffff, 89446c51007...","[89446124823ffff, 89446124827ffff, 8944612482b...","[89446124823ffff, 89446124827ffff, 8944612482b...",[89446124827ffff],"[89446124823ffff, 89446124827ffff, 8944612482b..."
8944612482bffff,"[8944612482bffff, 89446c51003ffff, 89446c51007...","[89446124823ffff, 89446124827ffff, 8944612482b...","[89446124823ffff, 89446124827ffff, 8944612482b...",[8944612482bffff],"[89446124823ffff, 89446124827ffff, 8944612482b..."
8944612482fffff,"[8944612482fffff, 89446c51003ffff, 89446c51007...","[89446124823ffff, 89446124827ffff, 8944612482b...","[89446124823ffff, 89446124827ffff, 8944612482b...",[8944612482fffff],"[89446124823ffff, 89446124827ffff, 8944612482b..."
89446124833ffff,"[89446124833ffff, 89446c51003ffff, 89446c51007...","[89446124823ffff, 89446124827ffff, 8944612482b...","[89446124823ffff, 89446124827ffff, 8944612482b...",[89446124833ffff],"[89446124823ffff, 89446124827ffff, 8944612482b..."


In [15]:
historic_crashes

,severity,commercial_vehicle,year,link_id,geometry,type
crash_point_id,,,,,,
17502452,O,False,2021,<NA>,POINT (-95.83075 29.74786),<NA>
17610265,O,False,2021,24691-24692,POINT (-95.43169 30.05207),<NA>
18038213,O,False,2021,22262-32253,POINT (-95.30201 29.81946),<NA>
18038491,C,False,2021,29278-29291,POINT (-94.78852 29.30099),<NA>
18038618,O,False,2021,<NA>,POINT (-95.24679 29.64548),<NA>
...,...,...,...,...,...,...
21443433,O,False,2025,30182-38025,POINT (-95.47112 29.78407),<NA>
21521008,O,False,2025,24287-30572,POINT (-95.34791 29.73435),<NA>
21543000,O,False,2025,20110-39886,POINT (-95.52802 29.66497),<NA>


In [9]:
# HistoricCrashes
print_status("HistoricCrashes")
historic_crashes = loader.load(context, "HistoricCrashes", category="PreprocessedConstants")
historic_crashes.head()


HistoricCrashes: 3 completed fields; 3 fields with missing values; 1 fields needing review


{'fields_with_missing_values': {'type': 100.0,
  'severity': 3.535,
  'link_id': 19.7043},
 'problematic_fields': {'type': 'source supplies crash_type_ped_cyclist and crash_type_intersection flags; 6,747 rows have both flags; single-category mapping unresolved; 100% missing values; nullable placeholder'}}

,severity,commercial_vehicle,year,link_id,geometry,type
crash_point_id,,,,,,
17502452,O,False,2021,<NA>,POINT (-95.83075 29.74786),<NA>
17610265,O,False,2021,24691-24692,POINT (-95.43169 30.05207),<NA>
18038213,O,False,2021,22262-32253,POINT (-95.30201 29.81946),<NA>
18038491,C,False,2021,29278-29291,POINT (-94.78852 29.30099),<NA>
18038618,O,False,2021,<NA>,POINT (-95.24679 29.64548),<NA>


In [10]:
# INRIX
print_status("INRIX")
inrix = loader.load(context, "INRIX", category="PreprocessedConstants")
inrix.head()


INRIX: 1 completed fields; 13 fields with missing values; 0 fields needing review


{'fields_with_missing_values': {'base_year_observed_volume_am': 94.387,
  'base_year_observed_volume_pm': 94.387,
  'base_year_observed_volume_md': 94.387,
  'base_year_observed_volume_ov': 94.387,
  'base_year_planning_time_index': 94.387,
  'base_year_observed_speed_am': 94.387,
  'base_year_observed_speed_pm': 94.387,
  'base_year_observed_speed_md': 94.387,
  'base_year_observed_speed_ov': 94.387,
  'base_year_reference_speed_am': 94.387,
  'base_year_reference_speed_pm': 94.387,
  'base_year_reference_speed_md': 94.387,
  'base_year_reference_speed_ov': 94.387},
 'problematic_fields': {}}

,base_year_planning_time_index,base_year_observed_volume_am,base_year_observed_volume_md,base_year_observed_volume_pm,base_year_observed_volume_ov,base_year_observed_speed_am,base_year_observed_speed_md,base_year_observed_speed_pm,base_year_observed_speed_ov,base_year_reference_speed_am,base_year_reference_speed_md,base_year_reference_speed_pm,base_year_reference_speed_ov
link_id,,,,,,,,,,,,,
5809-26997,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
5810-36183,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
5811-27007,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
5812-26990,1.082544,2369.0,4694.0,3486.0,3797.0,70.352738,68.568615,69.953984,67.764715,72.0,72.0,72.0,72.0
5813-24377,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [11]:
import geopandas as gpd


inrix_geom = gpd.GeoDataFrame(
    inrix.query("base_year_planning_time_index.notna()").merge(
        base_highway_links[["geometry"]], left_index=True, right_index=True, how="left"
    ),
    geometry="geometry"
)

kepler_visualize(inrix_geom)

User Guide: https://docs.kepler.gl/docs/keplergl-jupyter
Map saved to kepler.html!


In [12]:
# Projects
print_status("Projects")
projects = validate_table(context, "ScenarioUpstream", "Projects")
projects.head()


Projects: 98 completed fields; 0 fields with missing values; 0 fields needing review


{'fields_with_missing_values': {}, 'problematic_fields': {}}

,name,crash_improvement_projects,delay_reduction_projects,free_flow_speed_mph_changed_to,discounted_investment_cost,base_year_bike_trips,base_year_walk_trips,added_sidewalk_length_miles,added_sidewalk_width_feet,added_bike_amenity_length_miles,...,lodging_days_25_no_project,lodging_days_50_with_project,lodging_days_50_no_project,lodging_days_100_with_project,lodging_days_100_no_project,lodging_days_500_with_project,lodging_days_500_no_project,lodging_days_1000_with_project,lodging_days_1000_no_project,cell_ids
project_id,,,,,,,,,,,,,,,,,,,,,
1,0.0,[],[],30.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"[89446ca8d93ffff, 89446c35c07ffff, 89446c23443..."
2,0.0,[],[],30.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"[89446ca9013ffff, 89446c16127ffff, 89446cb916b..."
3,0.0,[],[],60.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"[89446c8a20bffff, 89446cae503ffff, 89446c30253..."
4,0.0,[],[],40.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"[89446ca1903ffff, 89446c27347ffff, 89446ca0503..."
5,0.0,[],[],40.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"[89446ca1903ffff, 89446d58007ffff, 89446caab03..."


In [13]:
# ScenarioLinks
print_status("ScenarioLinks")
scenario_links = validate_table(context, "ScenarioUpstream", "ScenarioLinks")
added = scenario_links["is_added_to_base"]
assert added.notna().all() and (added == ~scenario_links.index.isin(base_highway_links.index)).all()
print(f"{added.sum():,} of {len(added):,} scenario links are additions to the base network")
# Sample relationship coverage: every project must have links, with valid IDs and cells.
assigned = scenario_links["project_id"].notna()
assert scenario_links.loc[assigned, "project_id"].isin(projects.index).all()
project_link_counts = scenario_links["project_id"].value_counts().reindex(projects.index, fill_value=0)
assert project_link_counts.gt(0).all(), "Some sample projects have no assigned links"
for project_id, links in scenario_links.loc[assigned].groupby("project_id"):
    assert base_highway_links["cell_id"].reindex(links.index).isin(projects.loc[project_id, "cell_ids"]).all(), project_id
print(f"Synthetic sample: {assigned.sum():,} links assigned across {len(project_link_counts):,} projects")
display(project_link_counts.rename("assigned_link_count").sort_index().to_frame())
display(scenario_links.loc[assigned, ["project_id"]].sort_values("project_id").head(10))
scenario_links.head()


ScenarioLinks: 6 completed fields; 11 fields with missing values; 3 fields needing review


{'fields_with_missing_values': {'roadway_class': 4.7147,
  'facility_type': 4.7147,
  'lanes_count': 4.7147,
  'area_access_type': 7.108,
  'free_flow_speed_mph': 4.7147,
  'capacity': 4.7147,
  'cross_section_type': 4.7147,
  'is_in_1_mile_school_buffer': 4.7147,
  'is_in_100_year_flood_plain': 4.7147,
  'srts_status': 4.7147,
  'project_id': 99.8099},
 'problematic_fields': {'facility_type': 'tentative facility crosswalk; confirm NHS membership',
  'is_in_100_year_flood_plain': 'Needs reprocessing',
  'project_id': 'synthetic sample: three links per project selected within its sample cell_ids; other links intentionally unassigned'}}

3,800 of 47,341 scenario links are additions to the base network
Synthetic sample: 90 links assigned across 30 projects


,assigned_link_count
project_id,
1,3
2,3
3,3
4,3
5,3
6,3
7,3
8,3
9,3


,project_id
link_id,
21516-21796,1
21535-21583,1
21515-22741,1
22244-38007,2
22408-22752,2
22642-22639,2
20304-30222,3
20304-24193,3
23099-23504,3


,node_1_id,node_2_id,distance_miles,facility_type,roadway_class,lanes_count,area_access_type,free_flow_speed_mph,capacity,cross_section_type,is_in_100_year_flood_plain,is_in_1_mile_school_buffer,srts_status,geometry,is_removed_from_base,is_added_to_base,project_id
link_id,,,,,,,,,,,,,,,,,
5809-26997,5809,26997,1.24212,Non Interstate NHS,Fringe Rural,6,Rural Unrestricted,60.0,22800.0,Undivided,False,False,False,"LINESTRING (-96.05483 30.23059, -96.05801 30.2...",False,False,<NA>
5810-36183,5810,36183,1.12541,Other,Fringe Rural,2,Rural Unrestricted,45.0,3700.0,Undivided,False,False,False,"LINESTRING (-95.95032 30.23693, -95.94159 30.2...",False,False,<NA>
5811-27007,5811,27007,0.35634,Other,Fringe Rural,4,Rural Unrestricted,53.0,8450.0,Undivided,False,False,False,"LINESTRING (-95.82235 30.24426, -95.8184 30.24...",False,False,<NA>
5812-26990,5812,26990,3.32934,Interstate,Fringe Rural,3,Rural Restricted,64.0,51500.0,Divided,True,False,False,"LINESTRING (-96.18691 30.12929, -96.13169 30.1...",False,False,<NA>
5813-24377,5813,24377,0.3301,Other,Fringe Rural,2,Rural Unrestricted,53.0,4400.0,Undivided,True,False,False,"LINESTRING (-96.11122 30.04338, -96.10727 30.0...",False,False,<NA>


In [13]:
# ZoneAccessibilityMatrix
print_status("ZoneAccessibilityMatrix")
zone_accessibility = loader.load(context, "ZoneAccessibilityMatrix", category="ScenarioUpstream")
zone_accessibility.head()


ZoneAccessibilityMatrix: 4 completed fields; 0 fields with missing values; 0 fields needing review


{'fields_with_missing_values': {}, 'problematic_fields': {}}

,drive_15_min_taz_ids,drive_30_min_taz_ids,transit_45_min_taz_ids
taz_id,,,
1,"[1, 10, 100, 1000, 1003, 1004, 1005, 1006, 100...","[1, 10, 100, 1000, 1001, 1002, 1003, 1004, 100...","[1, 10, 100, 1000, 1001, 1002, 101, 1019, 102,..."
2,"[1, 10, 100, 1000, 1001, 1003, 1004, 1005, 100...","[1, 10, 100, 1000, 1001, 1002, 1003, 1004, 100...","[1, 10, 100, 1000, 1001, 1002, 101, 1019, 102,..."
3,"[1, 10, 100, 1000, 1003, 1004, 1005, 1006, 100...","[1, 10, 100, 1000, 1001, 1002, 1003, 1004, 100...","[1, 10, 100, 1000, 1001, 1002, 101, 1011, 1019..."
4,"[1, 10, 100, 1000, 1003, 1004, 1005, 1006, 100...","[1, 10, 100, 1000, 1001, 1002, 1003, 1004, 100...","[1, 10, 100, 1000, 1001, 1002, 101, 1019, 102,..."
5,"[1, 10, 100, 1000, 1003, 1004, 1005, 1006, 100...","[1, 10, 100, 1000, 1001, 1002, 1003, 1004, 100...","[1, 10, 100, 1000, 1001, 1002, 101, 1011, 1019..."


In [14]:
# NetworkOutput
print_status("NetworkOutput")
network_output = validate_table(context, "ScenarioUpstream", "NetworkOutput")
network_output.head()


NetworkOutput: 5 completed fields; 12 fields with missing values; 12 fields needing review


{'fields_with_missing_values': {'medium_truck_ratio_am': 100.0,
  'medium_truck_ratio_pm': 100.0,
  'medium_truck_ratio_md': 100.0,
  'medium_truck_ratio_ov': 100.0,
  'heavy_truck_ratio_am': 100.0,
  'heavy_truck_ratio_pm': 100.0,
  'heavy_truck_ratio_md': 100.0,
  'heavy_truck_ratio_ov': 100.0,
  'occupancy_am': 100.0,
  'occupancy_pm': 100.0,
  'occupancy_md': 100.0,
  'occupancy_ov': 100.0},
 'problematic_fields': {'medium_truck_ratio_am': '100% missing values; nullable placeholder',
  'medium_truck_ratio_pm': '100% missing values; nullable placeholder',
  'medium_truck_ratio_md': '100% missing values; nullable placeholder',
  'medium_truck_ratio_ov': '100% missing values; nullable placeholder',
  'heavy_truck_ratio_am': '100% missing values; nullable placeholder',
  'heavy_truck_ratio_pm': '100% missing values; nullable placeholder',
  'heavy_truck_ratio_md': '100% missing values; nullable placeholder',
  'heavy_truck_ratio_ov': '100% missing values; nullable placeholder',
  'occu

,volume_am,volume_md,volume_pm,volume_ov,geometry,medium_truck_ratio_am,medium_truck_ratio_pm,medium_truck_ratio_md,medium_truck_ratio_ov,heavy_truck_ratio_am,heavy_truck_ratio_pm,heavy_truck_ratio_md,heavy_truck_ratio_ov,occupancy_am,occupancy_pm,occupancy_md,occupancy_ov
link_id,,,,,,,,,,,,,,,,,
5809-26997,4922.3583,13738.7336,9941.137695,8496.749023,"LINESTRING (-96.05483 30.23059, -96.05801 30.2...",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
5810-36183,167.8401,394.6957,399.343262,314.546753,"LINESTRING (-95.95032 30.23693, -95.94159 30.2...",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
5811-27007,1433.235352,2763.418945,2045.7108,1733.604858,"LINESTRING (-95.82235 30.24426, -95.8184 30.24...",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
5812-26990,5457.900391,15153.0259,10658.238281,9842.2041,"LINESTRING (-96.18691 30.12929, -96.13169 30.1...",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
5813-24377,1231.0032,1850.7195,1563.89563,1095.2162,"LINESTRING (-96.11122 30.04338, -96.10727 30.0...",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [15]:
# CalibratedNetworkOutput
print_status("CalibratedNetworkOutput")
calibrated_network_output = validate_table(context, "KPIPreparation", "CalibratedNetworkOutput")
calibrated_network_output.head()


CalibratedNetworkOutput: 1 completed fields; 18 fields with missing values; 28 fields needing review


{'fields_with_missing_values': {'calibrated_vc_am': 4.7147,
  'calibrated_vc_pm': 4.7147,
  'calibrated_vc_md': 4.7147,
  'calibrated_vc_ov': 4.7147,
  'calibrated_truck_volume_am': 100.0,
  'calibrated_truck_volume_pm': 100.0,
  'calibrated_truck_volume_md': 100.0,
  'calibrated_truck_volume_ov': 100.0,
  'calibrated_truck_vmt_am': 100.0,
  'calibrated_truck_vmt_pm': 100.0,
  'calibrated_truck_vmt_md': 100.0,
  'calibrated_truck_vmt_ov': 100.0,
  'calibrated_travel_minutes_am': 100.0,
  'calibrated_travel_minutes_pm': 100.0,
  'calibrated_travel_minutes_md': 100.0,
  'calibrated_travel_minutes_ov': 100.0,
  'total_calibrated_daily_truck_volume': 100.0,
  'total_daily_truck_vmt': 100.0},
 'problematic_fields': {'calibrated_vc_am': 'synthetic from NetworkOutput (TestScenario); calibration pending; percent = 100 * calibrated volume / (ScenarioLinks capacity / period hours); unavailable or nonpositive capacities remain null; derivation reference: library-schema.json / CalibratedNetworkOut

,calibrated_volume_am,calibrated_truck_volume_am,calibrated_vmt_am,calibrated_truck_vmt_am,calibrated_travel_minutes_am,calibrated_vc_am,calibrated_volume_pm,calibrated_truck_volume_pm,calibrated_vmt_pm,calibrated_truck_vmt_pm,calibrated_travel_minutes_pm,calibrated_vc_pm,calibrated_volume_md,calibrated_truck_volume_md,calibrated_vmt_md,calibrated_truck_vmt_md,calibrated_travel_minutes_md,calibrated_vc_md,calibrated_volume_ov,calibrated_truck_volume_ov,calibrated_vmt_ov,calibrated_truck_vmt_ov,calibrated_travel_minutes_ov,calibrated_vc_ov,total_calibrated_daily_volume,total_calibrated_daily_vmt,total_calibrated_daily_truck_volume,total_daily_truck_vmt
link_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,
5809-26997,4922.3583,<NA>,6114.159827,<NA>,<NA>,64.767872,9941.137695,<NA>,12348.086228,<NA>,<NA>,174.405924,13738.7336,<NA>,17065.156158,<NA>,<NA>,361.545621,8496.749023,<NA>,10553.982131,<NA>,<NA>,409.930874,37098.978619,46081.384344,<NA>,<NA>
5810-36183,167.8401,<NA>,188.88892,<NA>,<NA>,13.608657,399.343262,<NA>,449.424884,<NA>,<NA>,43.172245,394.6957,<NA>,444.194472,<NA>,<NA>,64.004708,314.546753,<NA>,353.994049,<NA>,<NA>,93.5139,1276.425815,1436.502326,<NA>,<NA>
5811-27007,1433.235352,<NA>,510.719072,<NA>,<NA>,50.884095,2045.7108,<NA>,728.968568,<NA>,<NA>,96.838381,2763.418945,<NA>,984.716682,<NA>,<NA>,196.219097,1733.604858,<NA>,617.75274,<NA>,<NA>,225.676372,7975.969955,2842.157063,<NA>,<NA>
5812-26990,5457.900391,<NA>,18171.205983,<NA>,<NA>,31.793595,10658.238281,<NA>,35484.898837,<NA>,<NA>,82.782433,15153.0259,<NA>,50449.574963,<NA>,<NA>,176.540108,9842.2041,<NA>,32768.043612,<NA>,<NA>,210.221835,41111.368672,136873.723396,<NA>,<NA>
5813-24377,1231.0032,<NA>,406.354156,<NA>,<NA>,83.932036,1563.89563,<NA>,516.241947,<NA>,<NA>,142.17233,1850.7195,<NA>,610.922507,<NA>,<NA>,252.370841,1095.2162,<NA>,361.530868,<NA>,<NA>,273.80405,5740.83453,1895.049478,<NA>,<NA>


In [16]:
# CellLandUse — base-year placeholders
print_status("CellLandUse")
cell_land_use = loader.load(context, "CellLandUse", category="ScenarioUpstream")
cell_land_use.head()

CellLandUse: 1 completed fields; 9 fields with missing values; 9 fields needing review


{'fields_with_missing_values': {'imperviousness': 92.2763,
  'contains_transit': 100.0,
  'contains_retail': 92.2724,
  'population': 92.2724,
  'jobs': 92.2724,
  'households': 92.2724,
  'workers': 92.2724,
  'vulnerable_population': 92.2724,
  'e_vtol_compatability': 100.0},
 'problematic_fields': {'imperviousness': 'base-year sample used as scenario placeholder; scenario data pending',
  'contains_transit': '100% missing values; nullable placeholder',
  'contains_retail': 'base-year sample used as scenario placeholder; scenario data pending',
  'population': 'base-year sample used as scenario placeholder; scenario data pending',
  'jobs': 'base-year sample used as scenario placeholder; scenario data pending',
  'households': 'base-year sample used as scenario placeholder; scenario data pending',
  'workers': 'base-year sample used as scenario placeholder; scenario data pending',
  'vulnerable_population': 'base-year sample used as scenario placeholder; scenario data pending; Use la

,population,households,jobs,workers,contains_retail,vulnerable_population,imperviousness,contains_transit,e_vtol_compatability
cell_id,,,,,,,,,
89446124823ffff,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,<NA>,<NA>
89446124827ffff,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,<NA>,<NA>
8944612482bffff,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,<NA>,<NA>
8944612482fffff,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,<NA>,<NA>
89446124833ffff,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,<NA>,<NA>


In [17]:
# TAZDemographics — base-year placeholders
print_status("TAZDemographics")
taz_demographics = loader.load(context, "TAZDemographics", category="ScenarioUpstream")
taz_demographics.head()


TAZDemographics: 1 completed fields; 3 fields with missing values; 3 fields needing review


{'fields_with_missing_values': {'population': 92.6136,
  'jobs': 92.6136,
  'workers': 92.6136},
 'problematic_fields': {'population': 'base-year sample used as scenario placeholder; scenario data pending',
  'jobs': 'base-year sample used as scenario placeholder; scenario data pending',
  'workers': 'base-year sample used as scenario placeholder; scenario data pending'}}

,population,jobs,workers
taz_id,,,
1,NaN,NaN,NaN
2,NaN,NaN,NaN
3,NaN,NaN,NaN
4,NaN,NaN,NaN
5,NaN,NaN,NaN
